# Final-Layer Embedding Baseline — Community Alignment

Simplest usable baseline for embedding (prompt, candidate response) pairs from the Community Alignment choice-set data.

**Pipeline**
1. Load `choice_sets_ge10.parquet` (already pre-filtered upstream to `is_pregenerated_first_prompt == True` and choice sets with ≥10 annotators).
2. Validate required columns and drop rows with no preferred response.
3. Expand each row into 4 candidate rows (one per response A/B/C/D) and tag whether each was the preferred one.
4. Build a `text` field = `prompt + response`.
5. Run a frozen HF model, mean-pool the last hidden state with the attention mask, L2-normalize.
6. Mean-center and run PCA (default 32 components).
7. Save a candidate-level table with metadata + `pca1..pcaK`, the PCA object, and optionally the raw embeddings.
8. Show diagnostics + a `pca1` vs `pca2` scatter colored by preferred / not preferred.

**Schema note.** The user spec mentioned `is_pregenerated_first_prompt` and `first_turn_preferred_response`. The parquet we use is already filtered, and the preferred-response column is `chosen_response` (values `response_a`..`response_d`). This notebook adapts to that schema.

## Imports

In [1]:
from __future__ import annotations

import os
import gc
import math
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoTokenizer

## Config

All paths and hyperparameters live here. Override the env vars or edit inline.

In [2]:
# --- paths ---
DATA_PATH = Path(os.environ.get(
    "CA_DATA_PATH",
    "empirics_communityalignment/choice_sets_ge10.parquet",
))
OUTPUT_DIR = Path(os.environ.get("CA_OUTPUT_DIR", "empirics_communityalignment/final_layer_out"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_TABLE_PATH = OUTPUT_DIR / "candidates_with_pca.parquet"
PCA_PATH             = OUTPUT_DIR / "pca.joblib"
RAW_EMB_PATH         = OUTPUT_DIR / "embeddings.npy"

# --- model ---
MODEL_NAME    = os.environ.get("CA_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")
MAX_LENGTH    = int(os.environ.get("CA_MAX_LENGTH", 256))
BATCH_SIZE    = int(os.environ.get("CA_BATCH_SIZE", 32))
DEVICE        = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# --- pca / save ---
N_COMPONENTS  = int(os.environ.get("CA_N_COMPONENTS", 32))
SAVE_RAW_EMB  = os.environ.get("CA_SAVE_RAW_EMB", "1") == "1"

# --- sampling (set to None to use all rows) ---
MAX_ROWS = os.environ.get("CA_MAX_ROWS")
MAX_ROWS = int(MAX_ROWS) if MAX_ROWS else None

RANDOM_STATE = 0

print(f"device   : {DEVICE}")
print(f"model    : {MODEL_NAME}")
print(f"data     : {DATA_PATH}")
print(f"out dir  : {OUTPUT_DIR}")
print(f"n_comp   : {N_COMPONENTS}")

device   : cpu
model    : sentence-transformers/all-MiniLM-L6-v2
data     : empirics_communityalignment/choice_sets_ge10.parquet
out dir  : empirics_communityalignment/final_layer_out
n_comp   : 32


## Load and validate

In [3]:
REQUIRED_COLS = [
    "choice_set_id",
    "conversation_id",
    "annotator_id",
    "prompt_text",
    "chosen_response",
    "response_1",
    "response_2",
    "response_3",
    "response_4",
]

df = pd.read_parquet(DATA_PATH)
print(f"loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"missing required columns: {missing}")

if MAX_ROWS is not None and len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"subsampled to {len(df):,} rows for this run")

loaded: 14,231 rows x 20 cols


## Filter

`choice_sets_ge10.parquet` is already filtered to first-turn pregenerated prompts and to choice sets with ≥10 annotators, so we only drop rows with a missing preferred response.

In [4]:
before = len(df)
df = df[df["chosen_response"].notna()].copy()
df = df[df["chosen_response"].astype(str).str.strip() != ""].copy()
print(f"dropped {before - len(df):,} rows w/ null/empty chosen_response; kept {len(df):,}")

dropped 0 rows w/ null/empty chosen_response; kept 14,231


## Expand to candidate rows

Each annotator row → 4 candidate rows (A/B/C/D), each with the prompt + that candidate's response and an `is_preferred` flag.

In [5]:
LETTER_TO_COL = {"A": "response_1", "B": "response_2", "C": "response_3", "D": "response_4"}

def normalize_choice(x: str) -> str:
    """Map values like 'response_a', 'A', 'a' -> 'A'."""
    if not isinstance(x, str):
        return ""
    s = x.strip().upper()
    s = s.replace("RESPONSE_", "").replace("RESPONSE ", "")
    return s if s in LETTER_TO_COL else ""

df["chosen_letter"] = df["chosen_response"].map(normalize_choice)
bad = (df["chosen_letter"] == "").sum()
if bad:
    print(f"warning: {bad} rows had unrecognized chosen_response values; dropping")
    df = df[df["chosen_letter"] != ""].copy()

META_COLS = [
    c for c in [
        "choice_set_id", "conversation_id", "annotator_id", "wave", "assigned_lang",
        "annotator_age", "annotator_gender", "annotator_education_level",
        "annotator_political", "annotator_ethnicity", "annotator_country",
    ] if c in df.columns
]

long_rows = []
for letter, col in LETTER_TO_COL.items():
    sub = df[META_COLS + ["prompt_text", col, "chosen_letter"]].copy()
    sub = sub.rename(columns={col: "response_text"})
    sub["candidate_letter"] = letter
    sub["is_preferred"] = (sub["chosen_letter"] == letter).astype(int)
    long_rows.append(sub)

cand = pd.concat(long_rows, ignore_index=True)
cand = cand[cand["response_text"].notna()].copy()
cand = cand.reset_index(drop=True)
print(f"candidate rows: {len(cand):,}  (preferred: {cand['is_preferred'].sum():,})")
cand.head(2)

candidate rows: 56,924  (preferred: 14,231)


,choice_set_id,conversation_id,annotator_id,wave,assigned_lang,annotator_age,annotator_gender,annotator_education_level,annotator_political,annotator_ethnicity,annotator_country,prompt_text,response_text,chosen_letter,candidate_letter,is_preferred
0,e4bedcdd1e3ab6c2953b8e396ee09cd61b732d63,1315908176372549,61575530695320,1,hi,18-34,male,Some or complete graduate degree,I don't think of myself in this way,Indo-Aryan,india,मुझे फ़्रांस के पेरिस शहर में एक रोमांटिक गेटव...,"पेरिस शहर को प्यार का शहर कहा जाता है, और यहाँ...",D,A,0
1,5befa3474b45d7865fd76a4c89f0264c0dbf52ad,702992025732060,61575131153481,1,en,46-54,female,Post-secondary graduate,"Middle-of-the-road, centrist",Other,india,Can you give me some tips for choosing the per...,The key to choosing the perfect haircut for yo...,A,A,1


## Build text field

In [6]:
def build_text(prompt: str, response: str) -> str:
    p = (prompt or "").strip()
    r = (response or "").strip()
    return f"Prompt: {p}\n\nResponse: {r}"

cand["text"] = [build_text(p, r) for p, r in zip(cand["prompt_text"], cand["response_text"])]
print(cand["text"].str.len().describe().round(1))

count    56924.0
mean       721.2
std        293.9
min         75.0
25%        552.0
50%        690.0
75%        846.0
max       2911.0
Name: text, dtype: float64


## Load frozen model

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
model.to(DEVICE)

hidden_size = model.config.hidden_size
print(f"hidden size: {hidden_size}")

/Users/michellesi/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

hidden size: 384


## Extract embeddings

Mean-pool the last hidden layer using the attention mask; concatenate batches into one float32 array.

In [8]:
def mean_pool(last_hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.inference_mode()
def embed_texts(texts: List[str], batch_size: int = BATCH_SIZE) -> np.ndarray:
    out = np.empty((len(texts), hidden_size), dtype=np.float32)
    n_batches = math.ceil(len(texts) / batch_size)
    for i in range(n_batches):
        chunk = texts[i * batch_size:(i + 1) * batch_size]
        enc = tokenizer(
            chunk,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        outputs = model(**enc)
        pooled = mean_pool(outputs.last_hidden_state, enc["attention_mask"])
        out[i * batch_size:i * batch_size + pooled.shape[0]] = pooled.detach().cpu().numpy()
        if (i + 1) % 25 == 0 or (i + 1) == n_batches:
            print(f"  batch {i + 1}/{n_batches}")
    return out

texts = cand["text"].tolist()
print(f"embedding {len(texts):,} candidates...")
embeddings = embed_texts(texts)
print(f"embeddings shape: {embeddings.shape}")

del model, tokenizer
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

embedding 56,924 candidates...
  batch 25/1779


KeyboardInterrupt: 

## L2 normalize

In [ ]:
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.clip(norms, 1e-12, None)
embeddings = (embeddings / norms).astype(np.float32)
print(f"post-norm row norm range: [{np.linalg.norm(embeddings, axis=1).min():.4f}, "
      f"{np.linalg.norm(embeddings, axis=1).max():.4f}]")

## Mean-center + PCA

In [ ]:
n_comp = min(N_COMPONENTS, embeddings.shape[0], embeddings.shape[1])
if n_comp != N_COMPONENTS:
    print(f"clamping n_components: {N_COMPONENTS} -> {n_comp}")

emb_mean = embeddings.mean(axis=0, keepdims=True)
emb_centered = embeddings - emb_mean

pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
pcs = pca.fit_transform(emb_centered).astype(np.float32)
print(f"pca shape: {pcs.shape}")
print(f"explained variance (top 5): {pca.explained_variance_ratio_[:5].round(4)}")
print(f"cumulative explained: {pca.explained_variance_ratio_.sum():.4f}")

## Save outputs

In [ ]:
pca_cols = [f"pca{i + 1}" for i in range(pcs.shape[1])]
pca_df = pd.DataFrame(pcs, columns=pca_cols, index=cand.index)

keep_cols = META_COLS + [
    "prompt_text", "candidate_letter", "chosen_letter", "is_preferred", "response_text",
]
out = pd.concat([cand[keep_cols].reset_index(drop=True), pca_df.reset_index(drop=True)], axis=1)
out.to_parquet(CANDIDATE_TABLE_PATH, index=False)
print(f"saved candidate table -> {CANDIDATE_TABLE_PATH}  ({out.shape[0]:,} x {out.shape[1]})")

joblib.dump({"pca": pca, "mean": emb_mean, "model_name": MODEL_NAME, "n_components": n_comp}, PCA_PATH)
print(f"saved pca + mean      -> {PCA_PATH}")

if SAVE_RAW_EMB:
    np.save(RAW_EMB_PATH, embeddings)
    print(f"saved raw embeddings  -> {RAW_EMB_PATH}  shape={embeddings.shape}")

## Diagnostics

In [ ]:
print("candidate counts by is_preferred:")
print(out["is_preferred"].value_counts().rename({0: "not preferred", 1: "preferred"}))
print()
print("candidate counts by letter:")
print(out["candidate_letter"].value_counts().sort_index())
print()
print("pca summary (first 8 components):")
print(out[pca_cols[:8]].describe().round(4).T)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for label, color, alpha, size in [
    (0, "#888888", 0.25, 6),
    (1, "#d1495b", 0.55, 10),
]:
    mask = out["is_preferred"] == label
    ax.scatter(
        out.loc[mask, "pca1"],
        out.loc[mask, "pca2"],
        s=size, alpha=alpha, c=color,
        label="preferred" if label == 1 else "not preferred",
        linewidths=0,
    )
ax.set_xlabel("pca1")
ax.set_ylabel("pca2")
ax.set_title("Candidate embeddings (PCA), colored by preference")
ax.legend(loc="best", frameon=False)
fig.tight_layout()
plt.show()

## Sanity check — reload artifacts

In [ ]:
loaded = pd.read_parquet(CANDIDATE_TABLE_PATH)
loaded_pca = joblib.load(PCA_PATH)
print(f"reloaded table: {loaded.shape}")
print(f"reloaded pca  : n_components={loaded_pca['n_components']}, model={loaded_pca['model_name']}")